# Setting up the notebook and realtive paths

In [1]:
# Discover repo root and read all CSV files from the per-series folders
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path FIRST (before importing from functions)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Now import validation functions
from functions.validation_functions import (
    remove_duplicate_and_fill_missing,
 )

print('Repo root:', repo_root)

wiertsema_dir = repo_root / 'output_data' / 'wiertsema'
fugro_dir = repo_root / 'output_data' / 'fugro'
# Directory containing meteorological/stressor CSVs
stressor_dir = repo_root / 'input_stressors'
# Auto-detect stressor file paths (fallback to legacy names if none found)
stressor_files = list(stressor_dir.glob('*.csv'))
prec_candidates = sorted([p for p in stressor_files if 'prec' in p.name.lower() or 'rain' in p.name.lower()])
evap_candidates = sorted([p for p in stressor_files if 'evap' in p.name.lower() or 'makkink' in p.name.lower()])
precip_path = prec_candidates[0] if prec_candidates else (stressor_dir / 'knmi_berkhout_hourly_rain.csv')
evap_path = evap_candidates[0] if evap_candidates else (stressor_dir / 'knmi_berkhout_hourly_makkink.csv')
print('Detected stressor files:', len(stressor_files))
print('Using precip file ->', precip_path)
print('Using evap  file ->', evap_path)

out_fig = repo_root / 'output_data' / 'figures'
out_fig.mkdir(parents=True, exist_ok=True)

print('wiertsema_dir ->', wiertsema_dir)
print('fugro_dir    ->', fugro_dir)
print('precip_path ->', precip_path)
print('evap_path  ->', evap_path)

Repo root: d:\Users\jvanruitenbeek\data_validation
Detected stressor files: 3
Using precip file -> d:\Users\jvanruitenbeek\data_validation\input_stressors\prec_station_249_20260302113829.csv
Using evap  file -> d:\Users\jvanruitenbeek\data_validation\input_stressors\evap_station_249_20260302113829.csv
wiertsema_dir -> d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
fugro_dir    -> d:\Users\jvanruitenbeek\data_validation\output_data\fugro
precip_path -> d:\Users\jvanruitenbeek\data_validation\input_stressors\prec_station_249_20260302113829.csv
evap_path  -> d:\Users\jvanruitenbeek\data_validation\input_stressors\evap_station_249_20260302113829.csv


# Loading in the precipitation and evap file

Based on data with the "." seperator

For different dataset based on hourly data with the ";" and "," seperators

In [2]:
if precip_path.exists():
    df_prec_mm = pd.read_csv(precip_path,
                         sep=",",
                         decimal=".",
                         index_col=0,
                         parse_dates=[0],
                         encoding="utf-8-sig",
                         encoding_errors="replace")
else:
    raise FileNotFoundError(f'Precipitation file not found: {precip_path}')

if evap_path.exists():
    df_evap = pd.read_csv(evap_path,
                        sep=",",
                        decimal=".",
                        index_col=0,
                        parse_dates=[0],
                        encoding="utf-8-sig",
                        encoding_errors="replace")
else:
    raise FileNotFoundError(f'Evaporation file not found: {evap_path}')

In [3]:
df_prec_mm

,Precipitation
DATE,
2020-01-01,0.0
2020-01-02,0.0
2020-01-03,7.1
2020-01-04,0.5
2020-01-05,0.5
...,...
2026-02-22,8.9
2026-02-23,-0.1
2026-02-24,0.1


In [4]:
# df_evap.plot()

# Function to create the .csv files

In [5]:
# Configuration: Choose which dataset to process
# Options: 'fugro' or 'wiertsema'
dataset_choice = 'wiertsema'

# Set dataset root based on choice
if dataset_choice.lower() == 'fugro':
    dataset_root = fugro_dir
    print('Selected: FUGRO dataset')
elif dataset_choice.lower() == 'wiertsema':
    dataset_root = wiertsema_dir
    print('Selected: WIERTSEMA dataset')
else:
    raise ValueError("dataset_choice must be 'fugro' or 'wiertsema'")

print(f"Dataset root: {dataset_root}")

# Get all Stage-1 CSV files recursively: <origin>/only_csv/*.csv
csv_files = sorted(dataset_root.glob('*/only_csv/*.csv'))
print(f"Found {len(csv_files)} Stage-1 CSV files to process\n")

Selected: WIERTSEMA dataset
Dataset root: d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
Found 281 Stage-1 CSV files to process



In [6]:
# Process each file
results = []
failed_files = []

for i, file_path in enumerate(csv_files, start=1):
    try:
        print(f"[{i}/{len(csv_files)}] Processing: {file_path.name}")

        # Resolve origin stem from path: <dataset_root>/<origin>/only_csv/<series>.csv
        source_origin_stem = file_path.parent.parent.name

        # Loading in the stage-1 CSV
        validated_df = pd.read_csv(
            file_path,
            index_col=0,
            parse_dates=True,
            encoding="utf-8-sig",
            encoding_errors="replace"
        )

        print(validated_df.info())

        # Step 1: Remove duplicates and fill missing timestamps
        validated_df, _ = remove_duplicate_and_fill_missing(validated_df)

        # Align precipitation and evaporation to the validated data's index
        ref_index = validated_df.index
        prec_aligned = df_prec_mm.reindex(ref_index)
        evap_aligned = df_evap.reindex(ref_index)

        # Add precipitation and evapotranspiration columns
        validated_df["Precipitation"] = prec_aligned.iloc[:, 0].values
        validated_df["Evapotranspiration"] = evap_aligned.iloc[:, 0].values

        # Calculate recharge (Precipitation - Evapotranspiration)
        validated_df["recharge"] = validated_df["Precipitation"] - validated_df["Evapotranspiration"]

        # Add lineage fields for traceability
        validated_df["source_origin_stem"] = source_origin_stem
        validated_df["source_series_file"] = file_path.name

        # Ensure 'v0' remains last for readability
        all_cols = validated_df.columns.tolist()
        if 'v0' in all_cols:
            all_cols.remove('v0')
            validated_df = validated_df[all_cols + ['v0']]

        # Save to output: <dataset_root>/<origin>/knmi/<series>.csv
        output_dir = dataset_root / source_origin_stem / 'knmi'
        output_dir.mkdir(parents=True, exist_ok=True)
        output_file = output_dir / f"{file_path.stem}.csv"
        validated_df.to_csv(output_file, encoding="utf-8-sig")

        print(f"  ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“ Saved: {output_file}")
        print(f"    Final columns: {list(validated_df.columns)}\n")

        results.append({
            'Filename': file_path.name,
            'Origin': source_origin_stem,
            'Status': 'ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“',
            'Rows': len(validated_df),
            'Columns': len(validated_df.columns)
        })

    except Exception as e:
        print(f"  ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â Error: {str(e)}\n")
        failed_files.append(file_path.name)
        results.append({
            'Filename': file_path.name,
            'Origin': 'unknown',
            'Status': 'ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â',
            'Rows': 0,
            'Columns': 0
        })

# Summary
print("=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nSuccessful: {len(results_df[results_df['Status'] == 'ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“'])}/{len(csv_files)}")
if failed_files:
    print(f"Failed: {len(failed_files)}")
    for fname in failed_files[:10]:
        print(f"  - {fname}")
    if len(failed_files) > 10:
        print(f"  ... and {len(failed_files) - 10} more")

print(f"\nÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“ All processed CSVs saved under dataset root: {dataset_root}")

[1/281] Processing: 83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3951 entries, 2025-11-01 01:00:00 to 2026-04-28 00:00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   head    3951 non-null   float64
dtypes: float64(1)
memory usage: 61.7 KB
None
  ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“ Saved: d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema\83034-1\knmi\83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv
    Final columns: ['head', 'head_raw', 'Precipitation', 'Evapotranspiration', 'recharge', 'source_origin_stem', 'source_series_file', 'v0']

[2/281] Processing: 83034-1 HB002PB01 BE0049+00_BIKR_GMW_PB1_F-227.csv
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1343 entries, 2025-11-01 01:00:00 to 2025-12-26 23:00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  ---------